# Experiment 025 Attempt 006 acquisition analysis

This notebook regenerates the data-derived machine history, Attempt 005 postmortem, policy comparison, controller policy, and worker/controller source-boundary artifacts. The fleet analysis is a **PHYSICALLY GROUNDED MODEL**, not a physical swarm run. No provider mutation is performed.

In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd().resolve()
assert (repo_root / 'scripts' / 'experiment_025_attempt_006_analysis.py').is_file()
sys.path.insert(0, str(repo_root))
from scripts import experiment_025_attempt_006_analysis as analysis

run_root = repo_root / 'artifacts' / 'runs' / 'experiment-025-20260819T013016Z'
receipt = analysis.generate_all(run_root, iterations=1000, seed=25006)
receipt

{'status': 'NO_GO', 'decision': 'NO_GO_NO_PAID_INSTANCES', 'selected_policy': None, 'artifacts': {'history': {'path': 'C:\\Users\\Simon\\OneDrive\\Documents\\Python Scripts\\swarm-inference-lab\\artifacts\\runs\\experiment-025-20260819T013016Z\\preflight\\attempt-006-machine-reliability.json', 'sha256': '6913c0ca781359676e1469ce18b4270c16a7bddba6257a47012b379f6d794d0b'}, 'postmortem': {'path': 'C:\\Users\\Simon\\OneDrive\\Documents\\Python Scripts\\swarm-inference-lab\\artifacts\\runs\\experiment-025-20260819T013016Z\\preflight\\attempt-006-acquisition-postmortem.json', 'sha256': '4edb00b2ab405ee29afb7337ba770a7ec2a9cd0f5aab4980319dc417afaed80e'}, 'simulation': {'path': 'C:\\Users\\Simon\\OneDrive\\Documents\\Python Scripts\\swarm-inference-lab\\artifacts\\runs\\experiment-025-20260819T013016Z\\preflight\\attempt-006-acquisition-simulation.json', 'sha256': '8ba48d5ce1413221e48050e065f3339c70607d3f5ccaf574d1fa9da6027f3e27'}, 'policy': {'path': 'C:\\Users\\Simon\\OneDrive\\Documents\\Pyt

In [2]:
preflight = run_root / 'preflight'
postmortem = json.loads((preflight / 'attempt-006-acquisition-postmortem.json').read_text(encoding='utf-8'))
simulation = json.loads((preflight / 'attempt-006-acquisition-simulation.json').read_text(encoding='utf-8'))
policy = json.loads((preflight / 'attempt-006-acquisition-policy.json').read_text(encoding='utf-8'))
boundary = json.loads((preflight / 'attempt-006-source-boundary.json').read_text(encoding='utf-8'))

rows = []
for comparison in simulation['policy_comparisons']:
    if comparison['acquisition_window_seconds'] == 2400:
        base = comparison['scenario_results']['BASE']
        pessimistic = comparison['scenario_results']['PESSIMISTIC']
        rows.append({
            'policy': comparison['policy_id'],
            'base_probability': base['estimated_probability_full_simultaneous_readiness'],
            'pessimistic_probability': pessimistic['estimated_probability_full_simultaneous_readiness'],
            'median_total_cost_usd': base['active_storage_ingress_cost_usd']['median'],
            'median_peak_ready_roles': base['peak_ready_role_count']['median'],
            'credible': comparison['credible_path'],
        })
rows

[{'policy': 'attempt-005-current', 'base_probability': 0.0, 'pessimistic_probability': 0.0, 'median_total_cost_usd': 35.44848352882423, 'median_peak_ready_roles': 63.0, 'credible': False}, {'policy': 'continuous-ready-liveness', 'base_probability': 0.0, 'pessimistic_probability': 0.0, 'median_total_cost_usd': 35.47816724711676, 'median_peak_ready_roles': 64.0, 'credible': False}, {'policy': 'early-parallel-parent', 'base_probability': 0.0, 'pessimistic_probability': 0.0, 'median_total_cost_usd': 35.83983279026469, 'median_peak_ready_roles': 65.0, 'credible': False}, {'policy': 'live-alternate-refresh', 'base_probability': 0.0, 'pessimistic_probability': 0.0, 'median_total_cost_usd': 36.074825015149216, 'median_peak_ready_roles': 64.0, 'credible': False}, {'policy': 'reliability-aware-scoring', 'base_probability': 0.001, 'pessimistic_probability': 0.0, 'median_total_cost_usd': 32.8034776627618, 'median_peak_ready_roles': 81.0, 'credible': False}, {'policy': 'bounded-hedging', 'base_prob

In [3]:
assert postmortem['status'] == 'PASS'
assert postmortem['source_files']['canonical_event_log_sha256'] == analysis.ATTEMPT_005_EVENT_SHA256
assert postmortem['authoritative_result'] == 'INCOMPLETE'
assert postmortem['readiness']['total_ready_events'] == 56
assert postmortem['readiness']['peak_simultaneous_current_ready_roles'] == 46
assert simulation['iterations_per_policy_window_scenario'] == 1000
assert simulation['status'] == 'NO_GO'
assert simulation['selected_policy'] is None
assert policy['status'] == 'NO_GO'
assert policy['live_go_gate'] == 'NO_GO'
assert boundary['all_worker_executed_files_unchanged'] is True
assert policy['worker_image']['immutable_digest'] == analysis.WORKER_IMAGE_DIGEST
print('VALIDATED: NO_GO; no paid instances are justified by the offline evidence.')

VALIDATED: NO_GO; no paid instances are justified by the offline evidence.
